ALL KNC

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import mahalanobis
from scipy.stats import chi2

# Load the TSV file into a DataFrame
NM = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/PCA/0925/MD/KNC_1125_AADR_shrink.outlier.period.csv", sep=",", header=None, names=["ID", "PC1", "PC2", "PC3", "PC4", "pop"],skiprows=1)
#pop_filter = ["KNC_EIA", "KNC_EIA_o" ,"KNC_IA", "KNC_IA_o" , "NM_o","NM"]
pop_filter = ["KNC_EIA", "KNC_EIA_o" ,"KNC_IA", "KNC_IA_o", "KNC_LBA", "KNC_LBA_o"]
NM = NM[NM["pop"].isin(pop_filter)]

# Select PC1 and PC2 columns for the Mahalanobis calculation
NM_mahala = NM[["PC1", "PC2"]]

# Calculate the Mahalanobis distance for each row
mean_vector = NM_mahala.mean().values
cov_matrix = np.cov(NM_mahala.T)
inv_cov_matrix = np.linalg.inv(cov_matrix)

# Apply Mahalanobis distance calculation to each row
NM_mahala['mahala'] = NM_mahala.apply(
    lambda row: mahalanobis(row, mean_vector, inv_cov_matrix), axis=1
)

# Calculate the p-values for each Mahalanobis distance
df = 2  # degrees of freedom
NM_mahala['pvalue'] = 1 - chi2.cdf(NM_mahala['mahala']**2, df)

# Add ID column as index to match row names in R
NM_mahala.index = NM['ID']


# Display the resulting DataFrame
print(NM_mahala)

NM_mahala.to_csv("results_KNC_ALL_MD_060226.csv")
